**Live demo:** https://YOUR-APP-NAME.streamlit.app/  *(replace after deploying)*

# Titanic Passenger Survival Prediction System
AIC354 Assignment 2 · Task 1 · full Titanic dataset (891 passengers)

**Inputs:** `PClass`, `Gender`, `Sibling`, `Embarked` · **Target:** `Survived` (Yes/No)

In [ ]:
import pandas as pd, joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

## Phase 1: Data collection

In [ ]:
df = pd.read_csv('titanic.csv')   # full Titanic dataset, 891 passengers
print(df.shape)
df.head()

## Data understanding

In [ ]:
print(df[['survived','pclass','sex','sibsp','embarked']].isna().sum())   # embarked has missing values
print(df['survived'].value_counts())
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
df.groupby('sex')['survived'].mean().plot.bar(ax=ax[0], title='Survival rate by sex')
df.groupby('pclass')['survived'].mean().plot.bar(ax=ax[1], title='Survival rate by class')
plt.tight_layout(); plt.show()

## Data preprocessing
Keep 4 columns, drop the 2 rows with missing `embarked`, and turn the numeric codes into readable text categories.

In [ ]:
data = df[['pclass','sex','sibsp','embarked','survived']].dropna().copy()
data['PClass'] = data['pclass'].map({1:'First', 2:'Second', 3:'Third'})
data['Gender'] = data['sex'].str.capitalize()
data['Sibling'] = data['sibsp'].clip(upper=3).map({0:'Zero', 1:'One', 2:'Two', 3:'Three or more'})
data['Embarked'] = data['embarked'].map({'S':'Southampton', 'C':'Cherbourg', 'Q':'Queenstown'})
data['Survived'] = data['survived']   # already 0/1
FEATURES = ['PClass', 'Gender', 'Sibling', 'Embarked']
X, y = data[FEATURES], data['Survived']
print(data.shape); data[FEATURES + ['Survived']].head()

## Label encoding
Every text feature is converted to numbers with a fixed category order, inside the pipeline so the saved model accepts raw text.

In [ ]:
CATS = [['First','Second','Third'], ['Male','Female'], ['Zero','One','Two','Three or more'], ['Southampton','Cherbourg','Queenstown']]
encode = ColumnTransformer([('label', OrdinalEncoder(categories=CATS), FEATURES)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
print(encode.fit_transform(X_train)[:5])

## Phase 2 and 3: Model selection and training

In [ ]:
models = {'SVC': SVC(gamma='auto', probability=True, random_state=0),
          'Logistic Regression': LogisticRegression(),
          'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=0)}
fitted = {}
for name, m in models.items():
    fitted[name] = Pipeline([('encode', encode), ('model', m)]).fit(X_train, y_train)
    print(name, 'accuracy:', round(accuracy_score(y_test, fitted[name].predict(X_test)), 3))

## Phase 4: Evaluation
The assignment system uses the SVC, so it is evaluated in detail.

In [ ]:
model = fitted['SVC']
pred = model.predict(X_test)
print(classification_report(y_test, pred, target_names=['Not survived', 'Survived']))
ConfusionMatrixDisplay.from_predictions(y_test, pred); plt.show()

## Save the model and test a prediction

In [ ]:
joblib.dump(model, 'titanic_model.pkl')
new = pd.DataFrame([{'PClass': 'First', 'Gender': 'Female', 'Sibling': 'Zero', 'Embarked': 'Cherbourg'}])
loaded = joblib.load('titanic_model.pkl')
print('Prediction:', 'SURVIVED' if loaded.predict(new)[0] == 1 else 'NOT SURVIVED')